# अध्याय ७: च्याट अनुप्रयोगहरू निर्माण गर्दै
## OpenAI API द्रुत सुरुवात

यो नोटबुक [Azure OpenAI नमूना भण्डार](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) बाट अनुकूलित गरिएको हो जसले [Azure OpenAI](notebook-azure-openai.ipynb) सेवा पहुँच गर्ने नोटबुकहरू समावेश गर्दछ।

Python OpenAI API ले केही संशोधनहरूसँग Azure OpenAI मोडेलहरूसँग पनि काम गर्दछ। यहाँ थप भिन्दापनहरू बारे जान्नुस्: [Python सँग OpenAI र Azure OpenAI अन्त्यबिन्दुहरू बीच कसरी स्विच गर्ने](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# सिंहावलोकन  
"ठूला भाषा मोडेलहरू ती प्रकारका कार्यहरू हुन् जुन पाठलाई पाठमा नक्सा गर्छन्। दिइएका पाठको इनपुट स्ट्रिङबाट, एक ठूलो भाषा मोडेलले अर्को आउने पाठको भविष्यवाणी गर्ने प्रयास गर्छ"(1)। यो "छिटो सुरु" नोटबुकले प्रयोगकर्ताहरूलाई उच्च-स्तरका LLM अवधारणाहरू, AML सँग सुरु गर्न आवश्यक मुख्य प्याकेजहरू, प्रॉम्प्ट डिजाइनको सौम्य परिचय, र विभिन्न प्रयोग केसहरूको केही छोटो उदाहरणहरूलाई परिचय गराउनेछ। 


## सामग्री सूची  

[अवलोकन](#overview)  
[OpenAI सेवा कसरी प्रयोग गर्ने](#how-to-use-openai-service)  
[1. तपाइँको OpenAI सेवा सिर्जना गर्दै](#1.-creating-your-openai-service)  
[2. स्थापना](#2.-installation)    
[3. प्रमाणपत्रहरू](#3.-credentials)  

[प्रयोग केसहरू](#use-cases)    
[1. पाठ सारांश गर्नुहोस्](#1.-summarize-text)  
[2. पाठ वर्गीकरण गर्नुहोस्](#2.-classify-text)  
[3. नयाँ उत्पादन नामहरू सिर्जना गर्नुहोस्](#3.-generate-new-product-names)  
[4. क्लासिफायरलाई राम्रोसँग ट्यून गर्नुहोस्](#4.fine-tune-a-classifier)  

[सन्दर्भहरू](#references)


### तपाईंको पहिलो प्रॉम्प्ट निर्माण गर्नुहोस्  
यो छोटो अभ्यासले OpenAI मोडेलमा एउटा सरल कार्य "सारांश" को लागि प्रॉम्प्टहरू सबमिट गर्ने आधारभूत परिचय प्रदान गर्नेछ।  


**कदमहरू**:  
1. तपाईंको पाइथन वातावरणमा OpenAI लाइब्रेरी स्थापना गर्नुहोस्  
2. मानक सहयोगी लाइब्रेरीहरू लोड गर्नुहोस् र तपाईंले सिर्जना गरेको OpenAI सेवा को लागी तपाईंको सामान्य OpenAI सुरक्षा प्रमाणपत्रहरू सेट गर्नुहोस्  
3. तपाईंको कार्यका लागि एउटा मोडेल रोज्नुहोस्  
4. मोडेलको लागि एउटा सरल प्रॉम्प्ट बनाउनुहोस्  
5. तपाईंको अनुरोध मोडेल API मा पठाउनुहोस्!  


### 1. OpenAI स्थापना गर्नुहोस्


In [ ]:
%pip install openai python-dotenv

### २. सहायक पुस्तकालयहरू आयात गर्नुहोस् र प्रमाणपत्रहरू सुरु गर्नुहोस्


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### ३। सही मोडेल खोज्दै  
GPT-4o र GPT-4o मिनी जस्ता मोडेलहरूले प्राकृतिक भाषा बुझ्न र सिर्जना गर्न सक्छन्, र OpenAI प्लेटफर्ममा विभिन्न शक्ति र गतिका स्तरहरूसँग फरक फरक कार्यहरूका लागि उपलब्ध छन्।


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## ४. प्रॉम्प्ट डिजाइन  

"ठूला भाषा मोडेलहरूको जादू यो हो कि असीमित टेक्स्टमा यो भविष्यवाणी त्रुटि कम गर्न प्रशिक्षण दिएर, मोडेलहरूले यी भविष्यवाणीहरूका लागि उपयोगी अवधारणाहरू सिक्छन्। उदाहरणका लागि, तिनीहरूले यस प्रकारका अवधारणाहरू सिक्छन्"(1):

* कसरी हिज्जे लेख्ने
* कसरी व्याकरण काम गर्छ
* कसरी पाराफ़्रेज़ गर्ने
* कसरी प्रश्नको उत्तर दिने
* कसरी संवाद सञ्चालन गर्ने
* कसरी धेरै भाषाहरूमा लेख्ने
* कसरी कोड लेख्ने
* आदि।

#### ठूला भाषा मोडेललाई कसरी नियन्त्रण गर्ने  
"ठूला भाषा मोडेलमा सबैभन्दा प्रभावकारी इनपुट भनेको टेक्स्ट प्रॉम्प्ट हो(1)।

ठूला भाषा मोडेलहरू यसरी प्रॉम्प्ट गरेर आउटपुट उत्पादन गर्न सकिन्छ:

निर्देशन: मोडेललाई के चाहिन्छ भन्ने बताउनुहोस्
पूरक: तपाईँले चाहेको कुरा सुरु गर्न मोडेललाई प्रेरित गर्नुहोस्
प्रदर्शन: तपाईँले के चाहानुहुन्छ भनेर मोडेललाई देखाउनुहोस् जुन:
प्रॉम्प्टमा केही उदाहरणहरू
धेरै सय वा हजारौं उदाहरणहरू फाइन-ट्युनिङ प्रशिक्षण डेटासेटमा"



#### प्रॉम्प्ट बनाउन तीन आधारभूत दिशानिर्देशहरू छन्:

**देखाउनुहोस् र भन्नुहोस्**। तपाईँले के चाहनुहुन्छ स्पष्ट पार्नुहोस्, निर्देशनहरू, उदाहरणहरू वा दुवैको संयोजन मार्फत। यदि तपाईँ मोडेललाई कुनै वस्तुको सूचीलाई वर्णानुक्रमिक क्रममा क्रमबद्ध गर्न वा भावनाको आधारमा अनुच्छेद वर्गीकरण गर्न चाहनुहुन्छ भने, यसको लागि देखाउनुहोस्।

**गुणस्तरीय डेटा प्रदान गर्नुहोस्**। यदि तपाईँ वर्गीकारक बनाउँदै हुनुहुन्छ वा मोडेललाई कुनै नमुना पालन गराउन चाहनुहुन्छ भने, पर्याप्त उदाहरणहरू हुनुपर्छ भन्ने सुनिश्चित गर्नुहोस्। आफ्नो उदाहरणहरू प्रुफरीड गर्न नबिर्सनुहोस् — मोडेलले सामान्य हिज्जे त्रुटिहरू देख्न सक्छ र जवाफ दिन सक्छ, तर यसले यो जानाजानी भएको ठान्न सक्छ र यसले जवाफमा असर पुऱ्याउन सक्छ।

**सेटिङहरू जाँच गर्नुहोस्।** तापक्रम र top_p सेटिङहरूले मोडेलले जवाफ उत्पादन गर्दा कति निश्चित हुन्छ भनेर नियन्त्रण गर्छ। यदि तपाईँलाई एक मात्र सही जवाफ चाहिएको छ भने यी सेटिङहरू कम राख्नु ठीक हुन्छ। यदि तपाईँ फरक-फरक जवाफहरू खोज्दै हुनुहुन्छ भने यी उच्च राख्न सक्नुहुन्छ। यी सेटिङहरूमा सबैभन्दा ठुलो गल्ती हो यीलाई "बुद्धिमत्ता" वा "सृजनात्मकता" नियन्त्रण हो भनेर सोचेको।


स्रोत: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### ५. पठाउनुहोस्!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### एउटै कल फेरि दोहोर्याउँदा, परिणामहरू कसरी तुलना गरिन्छ? 


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## पाठ सारांश  
#### चुनौती  
पाठको अन्त्यमा 'tl;dr:' जोडेर पाठलाई सारांश बनाउनुहोस्। कसरी मोडेलले कुनै थप निर्देशन बिना धेरै कामहरू कसरी गर्ने बुझ्छ भन्ने कुरामा ध्यान दिनुहोस्। मोडेलको व्यवहार परिवर्तन गर्न र तपाईले प्राप्त गर्ने सारांशलाई अनुकूलन गर्न tl;dr भन्दा बढी वर्णनात्मक प्रॉम्प्टहरूसँग परीक्षण गर्न सक्नुहुन्छ(3)।  

हालैको कामले ठूलो पाठको संग्रहमा पूर्व-प्रशिक्षण गरी त्यसपछि विशिष्ट कार्यमा फाइन-ट्युनिङ गरेर धेरै NLP कार्यहरू र बेंचमार्कहरूमा ठूलो प्रगति देखाएको छ। सामान्यतया कार्य-एग्नोस्टिक आर्किटेक्चर भए तापनि, यस विधिले अझै पनि कार्य-विशिष्ट फाइन-ट्युनिङ डेटासेटहरू हजारौं वा दसौं हजार उदाहरणहरू आवश्यक पर्छ। यसको विपरीत, मानिसहरूले सामान्यतया केही उदाहरणहरूबाट वा सरल निर्देशनहरूबाट नयाँ भाषा कार्य गर्न सक्छन् - जुन वर्तमान NLP प्रणालीहरूले अझै धेरै संघर्ष गर्दैछ। यहाँ हामी देखाउँछौं कि भाषा मोडेलहरूलाई ठूलो बनाउनाले कार्य-एग्नोस्टिक, थोरै-शट प्रदर्शनमा ठूलो सुधार ल्याउँछ, कहिलेकाहीं पहिलेको अत्याधुनिक फाइन-ट्युनिङ विधिहरूसँग प्रतिस्पर्धा गर्नसम्म पुग्छ।  



सारांश  


# विभिन्न उपयोग केसहरूको लागि अभ्यासहरू  
1. पाठ सारांश बनाउने  
2. पाठ वर्गीकरण गर्ने  
3. नयाँ उत्पादन नामहरू उत्पन्न गर्ने  


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## पाठ वर्गीकरण  
#### चुनौती  
इन्फेरेन्सको समयमा प्रदान गरिएका वर्गहरूमा वस्तुहरू वर्गीकृत गर्नुहोस्। तलको उदाहरणमा, हामीले वर्गहरू र वर्गीकरण गर्नुपर्ने पाठ दुवै प्रम्प्टमा प्रदान गरेका छौं(*playground_reference)। 

ग्राहक सोधपुछ: नमस्कार, मेरो ल्यापटपको कीबोर्डका तल्लो मध्ये एक की भर्खरै टुट्यो र मलाई प्रतिस्थापन आवश्यक छ:

वर्गीकृत वर्ग:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## नयाँ उत्पादन नामहरू सिर्जना गर्नुहोस्
#### चुनौती
उदाहरण शब्दहरूबाट उत्पादन नामहरू सिर्जना गर्नुहोस्। यहाँ हामीले नामहरू उत्पन्न गर्न लागेका उत्पादनको बारेमा जानकारी प्रम्प्टमा समावेश गरेका छौं। हामीले प्राप्त गर्न चाहेको ढाँचा देखाउनको लागि एउटा समान उदाहरण पनि प्रदान गरेका छौं। थप रूपमा, हामीले उच्च र्‍याण्डमनेस र अधिक नवीन प्रतिक्रियाहरू प्राप्त गर्न तापक्रम मान उच्च राखेका छौं।

उत्पादन विवरण: घरमै बनाउन सकिने मिल्कशेक मेकर
बीउ शब्दहरू: छिटो, स्वस्थ, कम्प्याक्ट।
उत्पादन नामहरू: HomeShaker, Fit Shaker, QuickShake, Shake Maker

उत्पादन विवरण: एउटा जोड़ी जुत्ता जुन कुनै पनि खुट्टाको साइजसँग मेल खान सक्छ।
बीउ शब्दहरू: अनुकूलनीय, फिट, अॉम्नी-फिट।


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# सन्दर्भहरू  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [पाठ वर्गीकरण गर्नको लागि GPT मोडेलहरु फाइन-ट्यून गर्ने उत्तम अभ्यासहरू](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# थप मद्दतको लागि  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# सहभागीहरू
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
